# 2　大模型语义判定

第 7 项语义特征来自大语言模型的五级判定：把症状—病机对交给模型，按五个等级打分，
编码成 1.00／0.75／0.50／0.25／0.00。主用模型为 Qwen3.8-Flash，另外用 DeepSeek-V4-Flash
做一次对照判定。全库 28544 对逐对判一次；243 对参考样本再单独判一次，用来衡量模型调用的
批次稳定性。

判定结果已经冻结在 `数据/大模型判定/` 下，本文件默认直接读这些结果，不联网。
要重新调用模型时，先设好环境变量再跑 `rate_pairs`，密钥只从环境变量读。

In [ ]:
import os
from pathlib import Path

# 找到工程根目录（含「数据」文件夹的那一层）
ROOT = Path(os.path.abspath(".")).resolve()
while ROOT != ROOT.parent and not (ROOT / "数据").is_dir():
    ROOT = ROOT.parent
DATA = ROOT / "数据"
RES = ROOT / "结果"
RES.mkdir(exist_ok=True)

import json
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

LLM_DIR = DATA / "大模型判定"        # 大模型五级语义判定结果
GOLD_DIR = DATA / "专家标注"         # 专家评分记录与参考标注

LABEL_MAP = {"极强关联": 5, "强关联": 4, "中等关联": 3, "弱关联": 2, "极弱关联": 1}
KEY_COLS = ["症状", "病机"]

QWEN_MODEL, QWEN_BASE_URL, QWEN_KEY_ENV = (
    "qwen3.8-flash", "https://dashscope.aliyuncs.com/compatible-mode/v1", "DASHSCOPE_API_KEY")
DS_MODEL, DS_BASE_URL, DS_KEY_ENV = (
    "deepseek-v4-flash", "https://api.deepseek.com", "DEEPSEEK_API_KEY")
TEMPERATURE, MAX_WORKERS = 0.0, 8

# 解析顺序要先长后短，否则「强关联」会把「极强关联」也匹配走
PARSE_ORDER = ["极强关联", "极弱关联", "中等关联", "强关联", "弱关联"]

SYS_PROMPT = (
    "你是一位资深中医基础理论专家，精通中医症状学与病机学。"
    "请判断给定'症状'与'病机'之间的关联强度，分为五个等级，定义如下：\n"
    "极强关联：该病机几乎必然导致该症状，是该病机的核心/特征性表现；\n"
    "强关联：该病机常见且典型地导致该症状；\n"
    "中等关联：该病机可以导致该症状，但非特征性，或常需合并其他因素；\n"
    "弱关联：理论上可能相关，但临床少见或不典型；\n"
    "极弱关联：二者几乎无直接关联。\n"
    '仅输出JSON：{"level": "五档之一", "reason": "30字以内理由"}，不要输出其他内容。'
)


## 2.1　提示词、输出解析与调用

温度取 0.0，要求返回 JSON。解析时先按 JSON 里的 `level` 字段取档，取不到再在整段文本里
按「先长后短」的顺序找等级词。调用失败做指数退避重试；`response_format` 或
`extra_body` 不被支持时逐级退化，避免因为参数差异整批失败。

In [ ]:
def parse_level(text):
    """从模型输出解析五档等级，解析不出来返回 None。"""
    if not text:
        return None, ""
    reason = ""
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            reason = str(obj.get("reason", ""))[:100]
            for k in ("level", "等级", "关联强度"):
                if k in obj:
                    v = str(obj[k])
                    for lab in PARSE_ORDER:
                        if lab in v:
                            return LABEL_MAP[lab], reason
        except Exception:
            pass
    for lab in PARSE_ORDER:
        if lab in text:
            return LABEL_MAP[lab], reason
    return None, reason


def call_llm(client, model, s, p, disable_thinking=False, max_retry=4):
    """单对判定。失败做指数退避重试，参数不被支持时逐级退化。"""
    user_msg = f"症状：{s}\n病机：{p}\n请判断二者的关联强度等级。"
    kwargs = dict(model=model, temperature=TEMPERATURE, max_tokens=512,
                  response_format={"type": "json_object"})
    if disable_thinking:
        kwargs["extra_body"] = {"thinking": {"type": "disabled"}}
    for att in range(max_retry):
        try:
            r = client.chat.completions.create(
                messages=[{"role": "system", "content": SYS_PROMPT},
                          {"role": "user", "content": user_msg}], **kwargs)
            return r.choices[0].message.content
        except Exception as e:
            if att == 0 and "extra_body" in kwargs:
                kwargs.pop("extra_body")
                continue
            if "response_format" in kwargs:
                kwargs.pop("response_format")
                continue
            print(f"  [重试{att+1}] {s}-{p}: {type(e).__name__}")
            time.sleep(2 ** att)
    return None


def rate_pairs(pairs, provider="qwen", workers=MAX_WORKERS):
    """重新调用模型逐对判定。pairs 是 (症状, 病机) 的列表。"""
    from openai import OpenAI
    if provider == "qwen":
        model, url, env, think = QWEN_MODEL, QWEN_BASE_URL, QWEN_KEY_ENV, False
    else:
        model, url, env, think = DS_MODEL, DS_BASE_URL, DS_KEY_ENV, True
    key = os.environ.get(env, "").strip()
    if not key:
        raise RuntimeError("没有设置环境变量 %s。只复现论文数值的话，直接读冻结结果即可。" % env)
    client = OpenAI(api_key=key, base_url=url)

    def one(item):
        s, p = item
        lv, reason = parse_level(call_llm(client, model, s, p, think))
        return s, p, lv, reason

    rows = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for fut in as_completed([ex.submit(one, it) for it in pairs]):
            rows.append(fut.result())
    df = pd.DataFrame(rows, columns=KEY_COLS + ["llm_level", "理由"])
    df["llm_level"] = df["llm_level"].fillna(3).astype(int)     # 解析失败补中性档，理由保留
    df["llm_norm"] = (df["llm_level"] - 1) / 4.0
    return df

## 2.2　读出冻结的判定结果，得到第 7 项语义特征

全库 28544 对逐对判定的档位按 (档位 − 1) / 4 线性编码，作为第 7 项特征写进
`结果/全库语义特征.xlsx`。

In [ ]:
qall = pd.read_excel(LLM_DIR / "全库_语义判定_Qwen3.8-Flash.xlsx")
dall = pd.read_excel(LLM_DIR / "全库_语义判定_DeepSeek.xlsx")
q243 = pd.read_excel(LLM_DIR / "243对_语义判定_Qwen3.8-Flash.xlsx")
d243 = pd.read_excel(LLM_DIR / "243对_语义判定_DeepSeek.xlsx")
for d in (qall, dall, q243, d243):
    for c in KEY_COLS:
        d[c] = d[c].astype(str).str.strip()

for d in (qall, dall):
    d["llm_norm"] = (d["llm_level"].astype(float) - 1) / 4.0

print("全库判定：千问 %d 对，DeepSeek %d 对" % (len(qall), len(dall)))
print("千问档位分布:", qall["llm_level"].value_counts().sort_index().to_dict())
print("解析失败（补中性档）的对数 %d" % int(qall["理由"].isna().sum()))

qall[KEY_COLS + ["llm_level", "理由", "llm_norm"]].to_excel(RES / "全库语义特征.xlsx", index=False)
print("已写出 全库语义特征.xlsx")

## 2.3　243 对参考样本上的两批判定与批次稳定性（III-C）

同一批 243 对，一次混在全库 28544 对里判（全库判定批次），一次单独判（243 对独立判定批次）。
两次的差异反映的是模型调用的稳定性，不是模型能力。DeepSeek 的那一列也一并整理出来，
供 3 号文件做对照方法。

In [ ]:
gold = pd.read_excel(GOLD_DIR / "专家参考标注_243对.xlsx")
for c in KEY_COLS:
    gold[c] = gold[c].astype(str).str.strip()

merged = (gold[KEY_COLS + ["专家1", "专家2", "专家3", "y"]]
          .merge(qall[KEY_COLS + ["llm_level", "llm_norm"]]
                 .rename(columns={"llm_level": "qwen_all", "llm_norm": "qwen_all_norm"}),
                 on=KEY_COLS, how="left")
          .merge(q243[KEY_COLS + ["Qwen判定"]].rename(columns={"Qwen判定": "qwen_243"}),
                 on=KEY_COLS, how="left")
          .merge(d243[KEY_COLS + ["DeepSeek判定"]].rename(columns={"DeepSeek判定": "deepseek_243"}),
                 on=KEY_COLS, how="left"))
assert merged[["qwen_all", "qwen_243", "deepseek_243"]].notna().all().all(), "有缺失"
merged.to_excel(RES / "243对语义特征.xlsx", index=False)

a = merged["qwen_all"].values.astype(int)
b = merged["qwen_243"].values.astype(int)
df_stab = pd.DataFrame([
    ("精确一致率", round(float((a == b).mean()), 4), "%d/%d" % (int((a == b).sum()), len(a))),
    ("±1 级一致率", round(float((np.abs(a - b) <= 1).mean()), 4), ""),
    ("未加权 Cohen κ", round(float(cohen_kappa_score(a, b)), 4), ""),
    ("线性加权 κ", round(float(cohen_kappa_score(a, b, weights="linear")), 4), ""),
    ("二次加权 κ", round(float(cohen_kappa_score(a, b, weights="quadratic")), 4), "")],
    columns=["指标", "数值", "对数"])
df_stab.to_excel(RES / "语义特征批次稳定性.xlsx", index=False)
print(df_stab.to_string(index=False))
print("已写出 243对语义特征 / 语义特征批次稳定性")